# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjelmilan/flyrank-ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

###Rule

The baseline rule identifies content that should be refreshed by combining two signals: Days Since Refresh and Search Volume. Older content with higher search volume receives a higher priority score because updating it is more likely to improve visibility and user engagement.

A simple weighted score is calculated:

Score = (0.6 × Days Since Refresh) + (0.4 × Search Volume)

Items are ranked from highest to lowest score, and the highest-scoring pages are recommended for refresh.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# 1. Load Data
# -------------------------------------------------------------------------
url = "https://raw.githubusercontent.com/arjelmilan/flyrank-ai/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# -------------------------------------------------------------------------
# 2. Extract Signals using Dataset Column Names
# -------------------------------------------------------------------------
# Volume Signal: Using 90-day impressions
df["impressions_signal"] = df["impressions_90d"]

# Staleness Signal: Using days_since_last_update
df["days_stale"] = df["days_since_last_update"]

# CTR Signal
df["ctr_signal"] = df["ctr"]


df["expected_ctr"] = 1.0 / np.log2(df["avg_position"] + 2)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr_signal"]).clip(lower=0)

# ID Column Normalization for Deliverable CSV Format
df["client_hash_id"] = df["client_id"]
df["content_hash_id"] = df["content_id"]

# -------------------------------------------------------------------------
# 3. Compute Composite Non-Future Rule Score
# -------------------------------------------------------------------------
df["log_impressions"] = np.log1p(df["impressions_signal"])
df["staleness_years"] = df["days_stale"] / 365.0

# Rule Score Equation
df["rule_score"] = (
    (0.40 * df["log_impressions"]) +
    (0.35 * df["ctr_gap"] * 10) +
    (0.25 * np.minimum(df["staleness_years"], 2.0))
)

# -------------------------------------------------------------------------
# 4. Assign Reason Codes and Action Labels (ONE per row)
# -------------------------------------------------------------------------
def assign_action_and_reason(row):
    if row["days_stale"] > 180 and row["log_impressions"] > 5.0:
        return "REFRESH_CONTENT", "STALE_HIGH_IMPRESSION_DECAY"
    elif row["ctr_gap"] > 0.03 and row["impressions_signal"] > 500:
        return "OPTIMIZE_METADATA", "HIGH_IMPRESSIONS_LOW_CTR"
    elif row["days_stale"] > 270:
        return "UPDATE_FACTS_DATES", "STALE_EVERGREEN_CONTENT"
    else:
        return "NO_ACTION", "STABLE_PERFORMANCE"

df[["action_label", "reason_code"]] = df.apply(
    assign_action_and_reason, axis=1, result_type="expand"
)

# -------------------------------------------------------------------------
# 5. Rank the Queue & Export Output CSV
# -------------------------------------------------------------------------
ranked_queue = df.sort_values(by="rule_score", ascending=False).reset_index(drop=True)

export_cols = [
    "client_hash_id",
    "content_hash_id",
    "rule_score",
    "reason_code",
    "action_label"
]

output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "baseline_action_score.csv")

ranked_queue[export_cols].to_csv(output_file, index=False)

print(f"✅ Successfully exported ranked queue to: {output_file}")
print(f"📊 Total Processed Rows: {len(ranked_queue)}")
print("\n--- Action Summary Breakdown ---")
print(ranked_queue["action_label"].value_counts())
print("\n--- Top 5 Deliverable Queue Preview ---")
print(ranked_queue[export_cols].head())

✅ Successfully exported ranked queue to: ../outputs/baseline_action_score.csv
📊 Total Processed Rows: 30000

--- Action Summary Breakdown ---
action_label
NO_ACTION             19290
OPTIMIZE_METADATA     10651
REFRESH_CONTENT          32
UPDATE_FACTS_DATES       27
Name: count, dtype: int64

--- Top 5 Deliverable Queue Preview ---
      client_hash_id       content_hash_id  rule_score  \
0  client_d029fa3a95  content_8451fc6f034d    6.577567   
1  client_6208ef0f77  content_4a6607efcb46    6.430866   
2  client_4e07408562  content_8c19996aa890    6.357938   
3  client_f369cb89fc  content_d225ec9f3d46    6.354718   
4  client_19581e27de  content_7a6df559322d    6.297323   

                reason_code       action_label  
0  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_METADATA  
1  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_METADATA  
2  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_METADATA  
3  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_METADATA  
4  HIGH_IMPRESSIONS_LOW_CTR  OPTIMIZE_METADATA  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# -------------------------------------------------------------------------
# 1. Extract Top 20 Ranked Items
# -------------------------------------------------------------------------
top_20 = ranked_queue.head(20).copy()

# Print Top 20 Summary Table
print("=== TOP 20 RANKED QUEUE SUMMARY ===")
print(top_20[['content_hash_id', 'rule_score', 'action_label', 'reason_code', 'impressions_90d', 'days_since_last_update', 'ctr', 'avg_position']])

# -------------------------------------------------------------------------
# 2. Skeptic Audit Reasons & Failure Modes Map
# -------------------------------------------------------------------------
# Mapping template based on rule reason codes to fulfill "what would make it wrong"
failure_reasons_map = {
    "STALE_HIGH_IMPRESSION_DECAY": (
        "High impression volume + stale (>180d) content.",
        "High confidence due to massive search demand loss potential.",
        "Macro keyword search volume drop or seasonal demand shift rather than content staleness."
    ),
    "HIGH_IMPRESSIONS_LOW_CTR": (
        "Top-ranking page with large gap between expected and actual CTR.",
        "High confidence based on direct SERP click behavior.",
        "SERP layout features (e.g., AI Overview, Featured Snippets, Video Carousels) absorbing clicks."
    ),
    "STALE_EVERGREEN_CONTENT": (
        "Moderate volume page with no updates in >270 days.",
        "Medium confidence heuristic.",
        "Topic core facts remain unchanged or search intent hasn't shifted."
    ),
    "STABLE_PERFORMANCE": (
        "Fallback baseline rule classification.",
        "Low confidence override.",
        "Page is suffering from technical SEO bugs (301 redirect lag, mobile speed) not captured by content metadata."
    )
}

# Construct Audited Row Breakdown
audit_rows = []
for idx, row in top_20.iterrows():
    rank = idx + 1
    c_id = row['content_hash_id']
    action = row['action_label']
    reason = row['reason_code']

    why_flagged, confidence, wrong_reason = failure_reasons_map.get(
        reason,
        ("High rule composite score.", "Medium confidence.", "Unobserved external SERP changes or technical issues.")
    )

    audit_rows.append({
        "Rank": rank,
        "Content ID": c_id,
        "Action": action,
        "Reason Code": reason,
        "Why It's There": f"{why_flagged} (Imp: {int(row['impressions_90d'])}, Days Stale: {int(row['days_since_last_update'])})",
        "Confidence Note": confidence,
        "What Would Make It Wrong": wrong_reason
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

=== TOP 20 RANKED QUEUE SUMMARY ===
         content_hash_id  rule_score       action_label  \
0   content_8451fc6f034d    6.577567  OPTIMIZE_METADATA   
1   content_4a6607efcb46    6.430866  OPTIMIZE_METADATA   
2   content_8c19996aa890    6.357938  OPTIMIZE_METADATA   
3   content_d225ec9f3d46    6.354718  OPTIMIZE_METADATA   
4   content_7a6df559322d    6.297323  OPTIMIZE_METADATA   
5   content_5fe46e04994d    6.173757  OPTIMIZE_METADATA   
6   content_7247c9f3c142    6.062956  OPTIMIZE_METADATA   
7   content_e12868d1f396    6.052915  OPTIMIZE_METADATA   
8   content_36ff89c8214e    6.022150  OPTIMIZE_METADATA   
9   content_0022a6b4290f    5.974633  OPTIMIZE_METADATA   
10  content_c8e9d6ab9013    5.957004  OPTIMIZE_METADATA   
11  content_c84a0ab98e90    5.898088  OPTIMIZE_METADATA   
12  content_6973328a6bb8    5.861714  OPTIMIZE_METADATA   
13  content_73c54f78c06a    5.848559  OPTIMIZE_METADATA   
14  content_453722754fea    5.791307  OPTIMIZE_METADATA   
15  content_fea6a0d1

,Rank,Content ID,Action,Reason Code,Why It's There,Confidence Note,What Would Make It Wrong
0,1,content_8451fc6f034d,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
1,2,content_4a6607efcb46,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
2,3,content_8c19996aa890,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
3,4,content_d225ec9f3d46,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
4,5,content_7a6df559322d,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
5,6,content_5fe46e04994d,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
6,7,content_7247c9f3c142,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
7,8,content_e12868d1f396,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
8,9,content_36ff89c8214e,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."
9,10,content_0022a6b4290f,OPTIMIZE_METADATA,HIGH_IMPRESSIONS_LOW_CTR,Top-ranking page with large gap between expect...,High confidence based on direct SERP click beh...,"SERP layout features (e.g., AI Overview, Featu..."


## 4. Weak picks + leakage check



In [6]:
# -------------------------------------------------------------------------
# 1. Identify "Weak Picks" (Potential False Positives near the top)
# -------------------------------------------------------------------------
# Inspect top 30 items for potential edge cases
top_30 = ranked_queue.head(30)

# Case 1: High Impressions & High Staleness, BUT CTR is already excellent (Actual > Expected)
weak_picks_high_ctr = top_30[
    (top_30["action_label"] == "REFRESH_CONTENT") &
    (top_30["ctr"] > top_30["expected_ctr"])
]

# Case 2: High Impressions, BUT zero/near-zero search volume or zero clicks
weak_picks_zero_clicks = top_30[
    (top_30["clicks_90d"] == 0) &
    (top_30["impressions_90d"] > 1000)
]

print("=== 🔍 WEAK PICKS AUDIT ===")
print(f"Top 30 items flagged with strong CTR despite staleness: {len(weak_picks_high_ctr)}")
print(f"Top 30 items flagged with high impressions but ZERO clicks: {len(weak_picks_zero_clicks)}")

if not weak_picks_high_ctr.empty:
    print("\nSample Weak Pick (High CTR Stale Content):")
    display(weak_picks_high_ctr[["content_hash_id", "rule_score", "days_since_last_update", "ctr", "expected_ctr"]].head(2))

# -------------------------------------------------------------------------
# 2. Programmatic Data Leakage Guardrail
# -------------------------------------------------------------------------
# List of prohibited columns (future windows or label/target columns)
prohibited_columns = [
    "future_clicks", "future_impressions", "target", "label",
    "refresh_flag", "quick_win_flag", "ctr_fix_flag", "flyrank_flag"
]

# Columns actually used in score calculation
used_features = ["impressions_90d", "days_since_last_update", "ctr", "avg_position"]

leakage_detected = [col for col in used_features if col in prohibited_columns]

assert len(leakage_detected) == 0, f"🚨 LEAKAGE DETECTED! Prohibited features used: {leakage_detected}"

print("\n=== ✅ LEAKAGE CHECK PASSED ===")
print("• No future-window metrics (e.g., post-period traffic/clicks) were used.")
print("• No FlyRank product flags or ground-truth labels were used as rule inputs.")
print("• Scoring relies strictly on past observation signals (impressions_90d, days_since_last_update, ctr, avg_position).")


=== 🔍 WEAK PICKS AUDIT ===
Top 30 items flagged with strong CTR despite staleness: 0
Top 30 items flagged with high impressions but ZERO clicks: 1

=== ✅ LEAKAGE CHECK PASSED ===
• No future-window metrics (e.g., post-period traffic/clicks) were used.
• No FlyRank product flags or ground-truth labels were used as rule inputs.
• Scoring relies strictly on past observation signals (impressions_90d, days_since_last_update, ctr, avg_position).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.